# Exposure diagnostic

Two related questions, both about how exposure (`exposed`) enters the model:

**1. For non-zero rows: slope of `log(death_rate) ~ log(exposed)`.**
If deaths scale linearly with exposure (`deaths = rate · N`), the slope of `log(rate)` on `log(N)` should be ~0 — i.e., a fixed `log(N)` offset is appropriate for the bulk/tail rate models. A negative slope means deaths grow sub-linearly with N (larger populations have lower per-capita rates), and `log(N)` is better as a free covariate.

**2. For the S1 hurdle (`P(deaths >= 1)`): does `log(exposed)` predict it, controlling for covariates?**
More exposed population mechanically increases the chance any deaths occur. Fit a logistic regression to confirm and quantify.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

INPUT = Path('/mnt/team/idd/pub/idd_tc_mortality/00-data/current/input.parquet')
df = pd.read_parquet(INPUT)
df['log_exposed'] = np.log(df['exposed'])
df['has_deaths'] = (df['deaths'] >= 1).astype(int)

nz = df.loc[df['deaths'] >= 1].copy()
nz['death_rate'] = nz['deaths'] / nz['exposed']
nz['log_rate'] = np.log(nz['death_rate'])

print(f'Source: {INPUT.resolve()}')
print(f'Total rows: {len(df):,}   Non-zero death rows: {len(nz):,}   '
      f'P(deaths>=1) = {len(nz)/len(df):.1%}')

## 1. Slope of `log(death_rate) ~ log(exposed)` on non-zero rows

Slope ≈ 0 → offset assumption holds. Negative slope → sub-linear growth of deaths with exposure.

In [ ]:
sl, ic, r, p, se = stats.linregress(nz['log_exposed'], nz['log_rate'])
print(f'Overall   slope = {sl:.3f} ± {se:.3f}   r = {r:.3f}   p = {p:.2e}   n = {len(nz):,}')

In [ ]:
# Split bulk vs tail at several rate quantiles to see if the slope changes near the tail.
THRESHOLD_QUANTILES = [0.50, 0.75, 0.90]

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
axes = axes.flatten()

def fit_line(ax, x, y, color, label):
    if len(x) < 3:
        return
    sl, ic, r, p, _ = stats.linregress(x, y)
    xr = np.linspace(x.min(), x.max(), 200)
    ax.plot(xr, ic + sl * xr, color=color, linewidth=1.5,
            label=f'{label} slope={sl:.3f}  r={r:.2f}  n={len(x):,}')

axes[0].scatter(nz['log_exposed'], nz['log_rate'], s=8, alpha=0.4, color='steelblue')
fit_line(axes[0], nz['log_exposed'], nz['log_rate'], 'black', 'all')
axes[0].set_title('All non-zero rows')
axes[0].set_ylabel('log(death rate)')
axes[0].legend(fontsize=8)

for ax, q in zip(axes[1:], THRESHOLD_QUANTILES):
    threshold = float(np.quantile(nz['death_rate'], q))
    bulk = nz[nz['death_rate'] < threshold]
    tail = nz[nz['death_rate'] >= threshold]
    ax.scatter(bulk['log_exposed'], bulk['log_rate'], s=8, alpha=0.35, color='steelblue')
    ax.scatter(tail['log_exposed'], tail['log_rate'], s=8, alpha=0.35, color='firebrick')
    fit_line(ax, bulk['log_exposed'], bulk['log_rate'], 'steelblue', 'bulk')
    fit_line(ax, tail['log_exposed'], tail['log_rate'], 'firebrick', 'tail')
    ax.set_title(f'q = {q}   threshold = {threshold:.2e}')
    ax.set_xlabel('log(exposed)')
    ax.legend(fontsize=8)

fig.suptitle('log(death rate) vs log(exposed) — non-zero rows only', y=1.0)
fig.tight_layout(); plt.show()

## 2. Does `log(exposed)` predict `has_deaths`?

Logistic regression of `has_deaths` on `log_exposed` alone, then with covariates.

In [ ]:
X = sm.add_constant(df[['log_exposed']])
fit_uni = sm.Logit(df['has_deaths'], X).fit(disp=False)
print(fit_uni.summary())

df_sorted = df.sort_values('log_exposed')
pred = fit_uni.predict(sm.add_constant(df_sorted[['log_exposed']]))

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(df['log_exposed'], df['has_deaths'], alpha=0.15, s=10)
ax.plot(df_sorted['log_exposed'], pred, color='red', linewidth=2,
        label='predicted P(deaths>=1)')
ax.set_xlabel('log(exposed)'); ax.set_ylabel('has_deaths')
ax.set_title('P(deaths >= 1) vs log(exposed)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Full model: log_exposed + wind_speed + sdi + basin (one-hot) + is_island.
# Cast bool dummies to int — statsmodels rejects object dtype on bool columns.
covariates = df[['log_exposed', 'wind_speed', 'sdi', 'basin', 'is_island']].copy()
X_full = pd.get_dummies(covariates, columns=['basin'], drop_first=True).astype(float)
X_full = sm.add_constant(X_full)

fit_full = sm.Logit(df['has_deaths'], X_full).fit(disp=False)
print(fit_full.summary())

In [ ]:
# Compact view: coefficient, SE, p-value, odds ratio.
out = pd.DataFrame({
    'coef': fit_full.params,
    'se': fit_full.bse,
    'p_value': fit_full.pvalues,
    'odds_ratio': np.exp(fit_full.params),
})
out.round(4)